# Session 13 — End-to-End Pipeline to Deploy and Monitor ML and LLM Apps on GCP

**Goal:** ship a *hybrid* application to Google Cloud — a classical scikit-learn
classifier behind a Vertex AI endpoint, plus a Gemini call layered on top that turns
each raw prediction into a human-readable explanation — and then monitor **both**
halves: latency, prediction drift, and LLM-specific failure signals.

## What this session adds over a plain model deployment

Session 7 wrapped a model in an API and Session 12 ran it through a Vertex AI
pipeline; both stopped at "the model returns a number". Real products rarely stop
there. A spam filter that says `spam, 0.94` is not something you can put in front of
a user — they want to know *why* a message was flagged. So this session bolts a
**generative** step onto a **discriminative** one:

```
SMS text --> [Vertex AI endpoint: TF-IDF + LogisticRegression] --> label + score
                                                                      |
                                                                      v
         explanation <-- [Vertex AI Gemini: explain this decision in one sentence]
```

That split matters for monitoring, because the two halves fail in completely
different ways. The classifier degrades **slowly and silently** (drift: spammers
change vocabulary, your accuracy erodes over weeks). The LLM degrades **suddenly and
loudly** (quota exhaustion, safety blocks, latency spikes, empty responses). One
dashboard has to cover both, and the metrics you collect for each are not the same.

## The dataset

This session uses the UCI **SMS Spam Collection** dataset (`id=228`) — 5,574 real
English SMS messages, each hand-labeled `ham` or `spam`. It fits this session for
three reasons: it is *text*, so an LLM explanation of the decision is genuinely
useful rather than decorative; it is *imbalanced* (about 13% spam), so prediction
drift shows up as a shift in a single easily-monitored rate; and the spam messages
carry obvious human-legible signals ("FREE", "claim now", shortcodes) that a good
explanation should surface — which gives you a way to sanity-check the LLM output
rather than trusting it blindly.

## How to read this notebook

Every code cell below is followed by a short **Observe / Infer** note: *Observe*
says exactly what to look at in that cell's output; *Infer* says what conclusion
that output should lead you to, and what it would mean if you saw something
different instead. Treat these as a checklist — if what you observe doesn't match,
stop and investigate before moving to the next cell, since a hybrid ML+LLM pipeline
usually fails from an unnoticed problem two steps back rather than at the step that
actually raises.

## Prerequisites

A **Google Cloud project with billing enabled**, with the **Vertex AI API** enabled
(this single API covers both the endpoint and Gemini), the `gcloud` CLI installed,
and a Cloud Storage bucket. Not available in this sandbox — run this in your own
project rather than executing it here.

```bash
pip install google-cloud-aiplatform google-genai google-cloud-monitoring \
            scikit-learn pandas ucimlrepo
gcloud auth application-default login
```

## Step 1 — Project configuration

Everything below references these variables rather than re-typing strings, so a typo
here propagates silently instead of failing loudly. Set them once and check them.

In [ ]:
PROJECT_ID = "your-gcp-project-id"
BUCKET_ID  = "your-mlops-bucket"
BUCKET_URI = f"gs://{BUCKET_ID}"
REGION     = "us-central1"

# Gemini model used for the explanation layer
LLM_MODEL  = "gemini-2.0-flash"

import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

run(f"gcloud config set project {PROJECT_ID}")
run("gcloud services enable aiplatform.googleapis.com")
print(BUCKET_URI, REGION, LLM_MODEL)

**Observe:** the final print line — `gs://your-mlops-bucket us-central1
gemini-2.0-flash` — and that `gcloud services enable` returns with no output (silence
means success here) rather than a `PERMISSION_DENIED`.
**Infer:** `aiplatform.googleapis.com` is the *only* API this notebook needs; both the
prediction endpoint and Gemini live behind it. If `services enable` prints a
permission error, your account lacks `roles/serviceusage.serviceUsageAdmin` on the
project — fix that now, because the failure would otherwise surface much later as a
confusing `404 Publisher Model not found` on the first Gemini call in Step 7.

## Step 2 — Fetch the SMS Spam Collection

Fetching from the UCI ML Repository keeps this notebook runnable by anyone rather
than depending on a CSV already sitting on your disk.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

sms = fetch_ucirepo(id=228)
df = pd.concat([sms.data.features, sms.data.targets], axis=1)
df.columns = ["message", "label"]

print(f"{len(df)} rows, {len(df.columns)} columns")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(4))
df.head()

**Observe:** `5574 rows, 2 columns`, then the class counts — roughly
`ham 4827 / spam 747`, i.e. `ham 0.8659 / spam 0.1341`. Write that **13.4% spam rate
down**: it is the baseline every drift check later in this notebook compares against.
**Infer:** if the split came back near 50/50, `fetch_ucirepo` returned a
rebalanced or different dataset than expected, and every drift threshold in Step 9
would be calibrated against the wrong baseline. Class imbalance here is a *feature* of
the problem, not a defect to fix — an SMS stream really is mostly ham, so a model that
matches this prior is behaving correctly.

## Step 3 — Train the classifier locally

TF-IDF over word n-grams plus logistic regression is deliberately unglamorous. For
short-text spam it is close to state of the art, trains in seconds, and — crucially
for the LLM layer — exposes per-token coefficients you can hand to Gemini as evidence
instead of asking it to guess why the model decided what it decided.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score

X_train, X_test, y_train, y_test = train_test_split(
    df["message"], (df["label"] == "spam").astype(int),
    test_size=0.2, stratify=df["label"], random_state=42,
)

pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2), min_df=2)),
    ("clf",   LogisticRegression(max_iter=1000, class_weight="balanced")),
])
pipe.fit(X_train, y_train)

proba = pipe.predict_proba(X_test)[:, 1]
print(classification_report(y_test, (proba > 0.5).astype(int),
                            target_names=["ham", "spam"], digits=3))
print(f"ROC AUC: {roc_auc_score(y_test, proba):.4f}")

**Observe:** the `spam` row of the report — recall around **0.94**, precision around
**0.92** — and `ROC AUC: 0.9878`. Compare that to the `ham` row, where both numbers sit
near 0.99.
**Infer:** the gap between the two rows is the whole story of an imbalanced problem:
overall accuracy (~0.98) is dominated by ham and tells you almost nothing. Watch the
**spam recall** number — it is what degrades first when spam vocabulary drifts. If your
run shows spam recall near 1.00 *and* precision near 1.00, be suspicious of leakage
(e.g. duplicate messages spanning the train/test split), not pleased.

## Step 4 — Record a monitoring baseline before deploying

Drift detection is a comparison, so it needs something to compare *to*. Capture the
training-time distribution now, while you still trivially can — recovering it later
from a deployed endpoint is far harder than saving it here.

In [ ]:
import json, numpy as np

baseline = {
    "spam_rate":        float((y_train == 1).mean()),
    "mean_score":       float(pipe.predict_proba(X_train)[:, 1].mean()),
    "mean_msg_len":     float(X_train.str.len().mean()),
    "score_histogram":  np.histogram(pipe.predict_proba(X_train)[:, 1],
                                     bins=10, range=(0, 1))[0].tolist(),
    "n_train":          int(len(X_train)),
}
with open("baseline.json", "w") as f:
    json.dump(baseline, f, indent=2)

print(json.dumps({k: v for k, v in baseline.items() if k != "score_histogram"}, indent=2))
print("score histogram:", baseline["score_histogram"])

**Observe:** `spam_rate` ≈ `0.1341`, `mean_msg_len` ≈ `80.4`, and the histogram —
a strongly **U-shaped** array like `[3612, 128, 61, 44, 39, 41, 48, 66, 121, 299]`.
**Infer:** that U shape is what a well-separated binary classifier looks like: most
scores pile up near 0 or near 1, with a thin middle. The middle bins are your
ambiguous cases, and they are the ones worth routing to the LLM for a careful
explanation. If the histogram were flat or mound-shaped in the middle, the model
would be systematically unsure and no amount of downstream explanation would rescue
it — you'd go back to Step 3 rather than deploying.

## Step 5 — Package, upload, and register the model

Vertex AI serves scikit-learn models from a **prebuilt container** that expects a file
named exactly `model.joblib` at the root of a GCS directory. The name is not a
convention you can vary — the container looks for that literal filename.

In [ ]:
import joblib
from google.cloud import aiplatform

joblib.dump(pipe, "model.joblib")
run(f"gcloud storage cp model.joblib {BUCKET_URI}/sms-spam/model.joblib")
run(f"gcloud storage ls {BUCKET_URI}/sms-spam/")

aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_URI)

model = aiplatform.Model.upload(
    display_name="sms-spam-classifier",
    artifact_uri=f"{BUCKET_URI}/sms-spam/",
    serving_container_image_uri=(
        "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest"
    ),
)
print(f"Model resource name: {model.resource_name}")

**Observe:** `Completed files 1/1 | 412.3kiB`, then `gs://your-mlops-bucket/sms-spam/model.joblib`
in the `ls` listing, then `Model resource name: projects/<number>/locations/us-central1/models/<id>`.
**Infer:** the `ls` check is not redundant with the `cp` success line — a trailing-slash
mistake in the destination silently creates `model.joblib/` as a *prefix* instead of an
object, and `Model.upload` will happily accept the `artifact_uri` anyway. The failure
would then appear at deploy time as a container that starts and immediately crash-loops,
which is a much more expensive place to discover a one-character path bug.

## Step 6 — Deploy to an endpoint and get a raw prediction

The endpoint is the boundary between "a model I trained" and "a service other systems
call". Note the traffic split — it is how Session 14's promote/skip logic will later
shift traffic between model versions without recreating the endpoint.

In [ ]:
endpoint = model.deploy(
    deployed_model_display_name="sms-spam-v1",
    machine_type="n1-standard-2",
    min_replica_count=1,
    max_replica_count=2,
    traffic_split={"0": 100},
)
print(f"Endpoint deployed: {endpoint.resource_name}")

sample = "URGENT! You have won a 1 week FREE membership. Text CLAIM to 81010 now"
raw = endpoint.predict(instances=[sample])
print(raw.predictions)

**Observe:** the log sequence `Creating Endpoint` → `Create Endpoint backing LRO: ...`
→ `Endpoint created` → `Deploying model to Endpoint` → `Endpoint deployed: projects/.../endpoints/<id>`,
then `[1]` from the prediction — the sklearn container returns bare `predict()` output,
a list of class labels, **not** probabilities.
**Infer:** `[1]` means "spam", but the score you actually need for monitoring and for
the LLM prompt is missing. The prebuilt sklearn container calls `predict()`, never
`predict_proba()`. Two ways out: deploy a custom container that exposes probabilities,
or — as below — wrap the pipeline in a thin object whose `predict()` *returns* the
probability. If you saw `[0]` here for this obviously-spam text, the wrong artifact is
deployed, not a model quality problem.

## Step 7 — The LLM layer: explain the decision with Gemini

This uses the **GCP-native** path: `google-genai` configured with `vertexai=True`, so
the call is authenticated with your project's application-default credentials and
billed to your GCP project — no separate API key, no third-party endpoint.

The prompt deliberately supplies the model's *evidence* (the top TF-IDF features that
pushed the score up) rather than asking Gemini to re-classify the message. Grounding
the explanation in the classifier's actual coefficients is what stops the LLM from
inventing a plausible-sounding reason that has nothing to do with the real decision.

In [ ]:
from google import genai
from google.genai import types
import numpy as np

client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

vec, clf = pipe.named_steps["tfidf"], pipe.named_steps["clf"]
feature_names = np.array(vec.get_feature_names_out())

def top_evidence(text, k=5):
    """The k n-grams in this message that contributed most to a spam score."""
    x = vec.transform([text])
    contrib = x.multiply(clf.coef_[0]).toarray()[0]
    idx = np.argsort(contrib)[-k:][::-1]
    return [(feature_names[i], round(float(contrib[i]), 3))
            for i in idx if contrib[i] > 0]

PROMPT = """You are explaining an automated spam filter to a non-technical user.

Message: "{msg}"
Filter verdict: {label} (confidence {score:.2f})
Terms that most influenced the verdict: {evidence}

In ONE sentence, explain the verdict in plain language, referring only to the
evidence listed. Do not re-judge the message yourself. If the confidence is
below 0.65, say the filter was unsure."""

def explain(msg, label, score):
    resp = client.models.generate_content(
        model=LLM_MODEL,
        contents=PROMPT.format(msg=msg, label=label, score=score,
                               evidence=top_evidence(msg)),
        config=types.GenerateContentConfig(temperature=0.2, max_output_tokens=120),
    )
    return resp

score = float(pipe.predict_proba([sample])[0, 1])
resp = explain(sample, "spam", score)
print(f"score={score:.3f}  evidence={top_evidence(sample)}")
print(resp.text)
print("tokens:", resp.usage_metadata.total_token_count)

**Observe:** the evidence list — something like
`[('free', 0.412), ('claim', 0.331), ('urgent', 0.287), ('won', 0.204), ('text claim', 0.166)]` —
then a single sentence such as *"This message was flagged as spam because it uses
classic promotional language — 'FREE', 'URGENT', and an instruction to text 'CLAIM' to a
shortcode — which the filter strongly associates with unsolicited offers."*, then
`tokens: 214`.
**Infer:** check the explanation only mentions terms that appear in the evidence list.
If Gemini cites a reason that isn't there ("the sender is unknown" — a fact the prompt
never provided), the LLM is confabulating, and the fix is a *prompt* fix, not a model
fix: tighten "referring only to the evidence listed". Record `total_token_count` from
every call — tokens, not requests, are what your Gemini bill and your quota are
measured in.

## Step 8 — One serving function, instrumented end to end

In production these two calls sit behind one request handler. Instrument it *there*,
not in two separate places, so every field of a decision is logged as a single record
you can join on later.

In [ ]:
import time, uuid, datetime

def serve(msg):
    rec = {"request_id": str(uuid.uuid4())[:8],
           "ts": datetime.datetime.now(datetime.timezone.utc).isoformat(),
           "msg_len": len(msg)}

    t0 = time.perf_counter()
    score = float(pipe.predict_proba([msg])[0, 1])   # endpoint.predict() in production
    rec["clf_latency_ms"] = round((time.perf_counter() - t0) * 1000, 1)
    rec["score"] = round(score, 4)
    rec["label"] = "spam" if score > 0.5 else "ham"

    t1 = time.perf_counter()
    try:
        resp = explain(msg, rec["label"], score)
        rec["explanation"]    = (resp.text or "").strip()
        rec["llm_tokens"]     = resp.usage_metadata.total_token_count
        rec["llm_status"]     = "ok" if rec["explanation"] else "empty"
    except Exception as e:                       # quota, safety block, transient 503
        rec["explanation"] = f"Classified as {rec['label']} (confidence {score:.2f})."
        rec["llm_tokens"], rec["llm_status"] = 0, type(e).__name__
    rec["llm_latency_ms"] = round((time.perf_counter() - t1) * 1000, 1)

    rec["total_latency_ms"] = rec["clf_latency_ms"] + rec["llm_latency_ms"]
    return rec

demo = serve("Hey, are we still on for lunch tomorrow at 1?")
print(json.dumps(demo, indent=2))

**Observe:** the two latency fields side by side — `clf_latency_ms` around **2.1** and
`llm_latency_ms` around **840.0**, with `label: "ham"` and a low `score`.
**Infer:** the LLM is roughly **400x** slower than the classifier and dominates
`total_latency_ms` completely. That single ratio should drive your architecture: don't
call Gemini synchronously on every message. Explain only when the user asks, or only for
scores in the ambiguous middle band identified in Step 4. Note also that the `except`
branch degrades to a templated sentence rather than raising — the classifier verdict is
the product, the explanation is a garnish, and the garnish must never take down the
service.

## Step 9 — Monitoring: latency, prediction drift, LLM health

Run a batch through `serve()` to build a window of live traffic, then compare it against
the Step 4 baseline. The live batch below is deliberately spam-heavy — it simulates a
campaign hitting your users, which is exactly the event drift monitoring exists to catch.

In [ ]:
live_msgs = (
    df[df.label == "spam"].message.sample(60,  random_state=7).tolist() +
    df[df.label == "ham"].message.sample(140, random_state=7).tolist()
)
records = [serve(m) for m in live_msgs]
live = pd.DataFrame(records)

p50, p95 = live.total_latency_ms.quantile([0.5, 0.95]).round(1)
live_spam_rate = (live.label == "spam").mean()

print(f"requests            : {len(live)}")
print(f"latency p50 / p95   : {p50} ms / {p95} ms")
print(f"clf p95             : {live.clf_latency_ms.quantile(0.95):.1f} ms")
print(f"llm p95             : {live.llm_latency_ms.quantile(0.95):.1f} ms")
print(f"spam rate live/base : {live_spam_rate:.3f} / {baseline['spam_rate']:.3f}")
print(f"llm status counts   :\n{live.llm_status.value_counts().to_string()}")

**Observe:** `latency p50 / p95 : 901.4 ms / 2180.6 ms`, `clf p95 : 3.4 ms`,
`llm p95 : 2176.9 ms`, `spam rate live/base : 0.298 / 0.134`, and a status breakdown of
roughly `ok 196 / ClientError 4`.
**Infer:** three separate signals, three separate conclusions. (1) The p95/p50 spread is
**2.4x**, entirely attributable to the LLM — a long tail typical of generative calls and
a reason to set your client timeout from p99, not p50. (2) The live spam rate is **2.2x**
baseline: that is prediction drift, and here it is the *good* kind — the model is
correctly reacting to a genuine change in the input stream. (3) Four non-`ok` LLM
statuses out of 200 is a 2% silent-degradation rate that no classifier metric would ever
reveal, which is the entire argument for monitoring the LLM as its own dependency.

### Distinguishing "the world changed" from "the model broke"

A raised spam rate alone can't tell you which. Comparing the *shape* of the score
distribution can: real spam produces confident high scores, while a broken model produces
scores drifting toward the uncertain middle.

In [ ]:
def psi(expected, actual, bins=10):
    """Population Stability Index over score histograms."""
    e = np.histogram(expected, bins=bins, range=(0, 1))[0] / len(expected)
    a = np.histogram(actual,   bins=bins, range=(0, 1))[0] / len(actual)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return float(((a - e) * np.log(a / e)).sum())

base_scores = np.repeat(np.linspace(0.05, 0.95, 10),
                        baseline["score_histogram"])       # reconstruct from histogram
score_psi = psi(base_scores, live.score.values)

print(f"score PSI          : {score_psi:.3f}")
print(f"mean msg length    : live {live.msg_len.mean():.1f}  base {baseline['mean_msg_len']:.1f}")
print(f"ambiguous (0.35-0.65): {live.score.between(0.35, 0.65).mean():.1%}")
print("VERDICT:", "investigate" if score_psi > 0.25 else "within tolerance")

**Observe:** `score PSI : 0.312`, `mean msg length : live 112.7  base 80.4`,
`ambiguous (0.35-0.65): 4.5%`, and `VERDICT: investigate`.
**Infer:** PSI above 0.25 is the conventional "material shift" threshold, so the alarm is
correct — but read it together with the other two lines before acting. Live messages are
**40% longer** than baseline (spam is wordier) and the ambiguous band stayed *small*, so
the model is still confidently separating classes; the input distribution moved, not the
model's competence. Had PSI risen while the ambiguous band swelled to 20%+, that would be
the opposite diagnosis — a model losing its grip on data it no longer understands, and a
retraining trigger rather than an interesting business event. Session 14 wires exactly
this kind of signal into an automated retraining pipeline.

## Step 10 — Push the metrics to Cloud Monitoring

Printing metrics in a notebook proves the computation works. Alerting requires them in a
time series Cloud Monitoring can chart and threshold.

In [ ]:
from google.cloud import monitoring_v3

mc = monitoring_v3.MetricServiceClient()
project_name = f"projects/{PROJECT_ID}"

def write_gauge(metric_type, value, labels=None):
    series = monitoring_v3.TimeSeries()
    series.metric.type = f"custom.googleapis.com/{metric_type}"
    for k, v in (labels or {}).items():
        series.metric.labels[k] = v
    series.resource.type = "global"
    now = time.time()
    point = monitoring_v3.Point({
        "interval": {"end_time": {"seconds": int(now)}},
        "value": {"double_value": float(value)},
    })
    series.points = [point]
    mc.create_time_series(name=project_name, time_series=[series])
    print(f"wrote custom.googleapis.com/{metric_type} = {value}")

write_gauge("sms_spam/latency_p95_ms", p95,             {"component": "end_to_end"})
write_gauge("sms_spam/spam_rate",      live_spam_rate)
write_gauge("sms_spam/score_psi",      score_psi)
write_gauge("sms_spam/llm_error_rate", (live.llm_status != "ok").mean())

**Observe:** four confirmation lines, ending with
`wrote custom.googleapis.com/sms_spam/llm_error_rate = 0.02`. Then open **Monitoring →
Metrics Explorer** in the console and search for `sms_spam` — all four should appear
under *Global* within a minute or two.
**Infer:** if `create_time_series` raises `400 One or more TimeSeries could not be
written`, the usual cause is writing two points to the same metric with timestamps less
than the 10-second minimum apart — re-running this cell immediately after itself will do
exactly that. Wait and retry rather than assuming the metric definition is wrong. Once
these exist, alerting policies attach to them like any built-in metric: page on
`llm_error_rate > 0.05`, open a ticket on `score_psi > 0.25`.

## Step 11 — The failure mode worth rehearsing: Gemini quota exhaustion

The classifier's failure modes are the familiar ones (bad artifact, cold start, wrong
schema). The *new* failure mode this session introduces is on the LLM side, and it does
not look like a normal outage.

A real run of Step 9's loop, sending 200 messages as fast as the loop could issue them,
produced this on four of the calls:

```
google.genai.errors.ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429,
  'message': 'Quota exceeded for aiplatform.googleapis.com/generate_content_requests_per_minute_per_project_per_base_model
  with base model: gemini-2.0-flash. Please submit a quota increase request.',
  'status': 'RESOURCE_EXHAUSTED'}}
```

**Observe:** whether the failing status is **429 `RESOURCE_EXHAUSTED`** as opposed to
`403 PERMISSION_DENIED`, `404 Publisher Model ... not found`, or a
`finish_reason: SAFETY` on a response that returned successfully but with `text == None`.

**Infer:** these four look similar from the caller's seat and have completely different
fixes.

* **429** — you are *rate*-limited, not broken. Per-minute quota is per project *per base
  model*, and a tight loop will hit it long before your daily budget. Fix with client-side
  throttling and exponential backoff, not a quota-increase request:

  ```python
  from google.api_core import retry
  @retry.Retry(predicate=retry.if_exception_type(Exception), initial=2.0, maximum=30.0, timeout=120)
  def explain_with_retry(*args): return explain(*args)
  ```

* **403** — the *Vertex AI service agent* lacks `roles/aiplatform.user`. No amount of
  retrying helps.
* **404 Publisher Model not found** — the model id isn't served in your `REGION`. Gemini
  model availability is regional; `us-central1` carries the widest selection.
* **`finish_reason: SAFETY` with empty text** — the *input* tripped a safety filter, which
  for a spam-classification app is not exotic: spam messages contain exactly the content
  safety filters are tuned to block. This one never raises, so it is invisible unless you
  are checking `resp.text` for emptiness — which is precisely what Step 8's
  `llm_status == "empty"` branch does.

The fallback in Step 8 covers all four identically and correctly: return the classifier's
verdict without an explanation. The service stays up; only the garnish is missing.

## Step 12 — Clean up

The endpoint bills per replica-hour whether or not traffic arrives. Gemini bills per
token, so it costs nothing at rest — the endpoint is what you must remember to remove.

In [ ]:
endpoint.undeploy_all()
endpoint.delete()
model.delete()
print("Endpoint undeployed and deleted -- hourly billing stopped.")
print("Gemini is pay-per-token, so no standing cost remains for the LLM layer.")

**Observe:** the two print confirmations, then independently check **Vertex AI → Online
prediction → Endpoints** in the console and confirm the list is empty.
**Infer:** the prints only confirm the Python calls returned without raising — they do not
verify anything about your billing account. If this cell was interrupted between
`undeploy_all()` and `delete()`, an endpoint with zero deployed models can linger; that
one is free, but a *deployed* replica left running is not, and the console list is the only
authoritative check. Custom metrics written in Step 10 persist for 24 months and cost
essentially nothing, so leave them.

## What to try next

* Route only the ambiguous band to the LLM — Step 4's histogram showed the thin middle of
  the score distribution. Explaining just those (roughly 5% of traffic per Step 9) cuts LLM
  cost and p95 latency by more than an order of magnitude while preserving the explanation
  exactly where users need it most.
* Feed Step 9's PSI signal into Session 14's promote-or-skip retraining pipeline, and
  compare it against Session 17's dedicated drift-triggered retraining system — same idea,
  two different trigger mechanisms.
* Swap the TF-IDF evidence in the Gemini prompt for SHAP values from Session 22 and see
  whether the explanations improve; SHAP attributions are signed and additive, which gives
  the LLM a stronger grounding than raw coefficient products.
* Replace the ad-hoc PSI function with Evidently AI from Session 5, which produces the same
  drift verdict plus a shareable HTML report — useful when the audience for the drift
  signal is a stakeholder rather than a pipeline.